# 114. RG3 미성형 사례-정상 상대거리 앙상블
113번의 내부 기준 일반화 실패를 확인한 뒤 다른 학습 방식으로 변경. 574개/미성형8 고정, 외부구간 전체 제외. 정상 기준 RobustScaler, 실제 학습 미성형/정상 근접거리의 비율. 가스·CN7·미래센서 제외. 이미 본 자료의 개발평가이지 신규 독립검증 아님.
각6개 표현 raw/phase/thermal/sequence, 이웃1/3 및 평균 전문가. 학습 내부에서 조합과 F1 임계값 선택. 정상99% 기준은 비교용으로 별도 보고.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import pairwise_distances,confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);groups=d.comparison_block.to_numpy();blocks=np.unique(groups)
assert len(d)==574 and y.sum()==8
sensors=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Filling_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Speed']
ratio=lambda a,b:a/b.where(b.abs()>1e-6)
views={'raw6':d[sensors],
 'phase6':pd.DataFrame({'pressure_gap':d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure,'fill_fraction':ratio(d.Filling_Time,d.Injection_Time),'cushion_fraction':ratio(d.Cushion_Position,d.Plasticizing_Position),'meter_cushion_delta':d.Plasticizing_Position__delta1-d.Cushion_Position__delta1,'thermal_gap':d.Barrel_Temperature_5-(d.Mold_Temperature_3+d.Mold_Temperature_4)/2,'fill_per_speed':ratio(d.Filling_Time,d.Max_Injection_Speed)}),
 'thermal6':d[['Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4','Plasticizing_Time']]}
sq=pd.read_csv(t/'rg3_shortshot_sequence_features.csv',parse_dates=['TimeStamp']).set_index('TimeStamp')
views['sequence6']=sq.loc[d.TimeStamp,['Max_Injection_Pressure_path_efficiency','Cushion_Position_path_efficiency','Barrel_Temperature_5_path_efficiency','pressure_recent3_energy_share','pressure_cushion_delta_sync','pressure_then_cushion_lag_advantage']].reset_index(drop=True)
names=[name+'_k'+str(k) for name in views for k in [1,3]]
memo={}
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 normal=ref[y[ref]==0];positive=ref[y[ref]==1]
 if not len(positive):return np.zeros((len(q),len(names)))
 scores=[]
 for name,x in views.items():
  a=x.replace([np.inf,-np.inf],np.nan).to_numpy(float)
  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal]);a=imp.transform(a)
  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12)
  dn=np.sort(pairwise_distances(a[q],a[normal]),axis=1)
  dp=np.sort(pairwise_distances(a[q],a[positive]),axis=1)
  for k in [1,3]:
   nd=dn[:,:min(k,len(normal))].mean(axis=1);pdistance=dp[:,:min(k,len(positive))].mean(axis=1)
   scores.append(nd/(nd+pdistance+1e-8))
 out=np.column_stack(scores);memo[key]=out;return out
weights=[(n,np.eye(len(names))[j]) for j,n in enumerate(names)]
weights += [('all_view_mean',np.ones(len(names))/len(names))]
weights += [(v+'_k_mean',np.array([.5 if n.startswith(v+'_') else 0 for n in names])) for v in views]
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b);out=predict(ref,q)
 inner=np.full((len(ref),len(names)),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for name,w in weights:
  score=out@w;cal=inner@w;cut,fi=threshold(y[ref],cal)
  candidates.append((fi,name,w,cut))
  for rule,th in [('inner_F1',cut),('normal99',float(np.quantile(cal[y[ref]==0],.99,method='higher')))]:
   pr=score>=th
   for j,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 best=max(candidates,key=lambda z:(z[0],-np.count_nonzero(z[2]),z[1]))
 fi,name,w,th=best;score=out@w;pr=score>=th
 choices.append(dict(block=b,selected=name,inner_F1=fi,threshold=th))
 for j,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
p.to_csv(t/'rg3_shortshot_case_contrast_predictions.csv',index=False,encoding='utf-8-sig')
m.to_csv(t/'rg3_shortshot_case_contrast_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(t/'rg3_shortshot_case_contrast_choices.csv',index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False))
print('PRIMARY',m[m.model.eq('primary_inner_selected')].to_string(index=False))


                 model     rule  TP  FP  FN  TN       F1       AP
               raw6_k1 normal99   1   7   7 559 0.125000 0.026655
           thermal6_k1 inner_F1   1  48   7 518 0.035088 0.019838
             phase6_k1 inner_F1   3 185   5 381 0.030612 0.018511
               raw6_k3 inner_F1   6 428   2 138 0.027149 0.012420
       thermal6_k_mean inner_F1   2 142   6 424 0.026316 0.020840
          sequence6_k3 inner_F1   4 349   4 217 0.022161 0.011244
         phase6_k_mean inner_F1   2 178   6 388 0.021277 0.016923
               raw6_k1 inner_F1   2 199   6 367 0.019139 0.026655
             phase6_k3 inner_F1   2 225   6 341 0.017021 0.015512
         all_view_mean inner_F1   2 237   6 329 0.016194 0.016314
           thermal6_k3 inner_F1   1 115   7 451 0.016129 0.024638
           raw6_k_mean inner_F1   2 264   6 302 0.014599 0.013572
          sequence6_k1 inner_F1   2 267   6 299 0.014440 0.012413
      sequence6_k_mean inner_F1   2 284   6 282 0.013605 0.011790
         p